In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
## Create a function that reads and cleans the data
def tidy_pokedata():
    ## Read the data
    data_newDB = pd.read_csv("../data/pokemonDB_dataset.csv")
    data_pokedex = pd.read_csv("../data/All_Pokemon.csv")

    ## Inner join on the datasets by name
    merged_data = pd.merge(data_newDB, data_pokedex, left_on="Pokemon", right_on="Name", how="inner")

    ## Drop unecessary columns (duplicates)
    data_reduced = merged_data.drop(columns=
                                    ['Type', 'Height_x', 'Weight_x', 'Abilities_x', 'HP', 'Att', 'Def',
                                 'Spa', 'Spd', 'Spe', 'Experience type', 'Catch Rate_x', 'Pokemon'])
    ## Rename columns
    data_renamed = data_reduced.rename({'Abilities_y' : 'Abilities',
                                        'Catch Rate_y' : 'Catch Rate',
                                        'Height_y' : 'Height',
                                        'Weight_y' : 'Weight',
                                        'Number' : 'National Dex Number',
                                        'Type 1' : 'Type1',
                                        'Type 2' : 'Type2'}, axis=1)

    ## Put 'Name' as the first column in the dataframe
    data_renamed.insert(0, 'Name', data_renamed.pop('Name'))

    return data_renamed, data_newDB, data_pokedex

pokedata, _, _, = tidy_pokedata()
pokedata.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 63 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Name                     975 non-null    str    
 1   Species                  975 non-null    str    
 2   EV Yield                 975 non-null    str    
 3   Base Friendship          975 non-null    str    
 4   Base Exp                 975 non-null    str    
 5   Growth Rate              975 non-null    str    
 6   Egg Groups               975 non-null    str    
 7   Gender                   975 non-null    str    
 8   Egg Cycles               975 non-null    str    
 9   HP Base                  975 non-null    int64  
 10  HP Min                   975 non-null    int64  
 11  HP Max                   975 non-null    int64  
 12  Attack Base              975 non-null    int64  
 13  Attack Min               975 non-null    int64  
 14  Attack Max               975 non-null

### Replicate the National Dex
Create a dataframe that indexes the pokemon based on their National Dex Number, and does not include any Mega forms or regional variants.

In [3]:
## First address the Mega and Regional Pokemon
mega_pkmn = pokedata.loc[pokedata['Mega Evolution'] == 1]
reg_pkmn = pokedata.loc[(pokedata['Alolan Form'] == 1) | (pokedata['Galarian Form'] == 1)] # | = bitwise OR operator

## We could filter the NatDex dataframe to exclude these.
condition_no_specials = (pokedata['Mega Evolution'] != 1) & (pokedata['Alolan Form'] != 1) & (pokedata['Galarian Form'] != 1)
Trial_NatDex = pokedata.loc[condition_no_specials] # but there's a better way

In [4]:
## There may be more duplicates in the National Dex Number. A more general way to remove duplicates is as follows:
NDN_duplicates_bool = pokedata.duplicated(subset=['National Dex Number'])
NDN_duplicates = pokedata.loc[NDN_duplicates_bool]

NDN_duplicates.head()

,Name,Species,EV Yield,Base Friendship,Base Exp,Growth Rate,Egg Groups,Gender,Egg Cycles,HP Base,...,Against Bug,Against Rock,Against Ghost,Against Dragon,Against Dark,Against Steel,Against Fairy,Height,Weight,BMI
1,Mega Abomasnow,Frost Tree Pokémon,"1 Attack, 1 Sp. Atk",50 (normal),208,Slow,"Grass, Monster","50% male, 50% female","20 (4,884–5,140 steps)",90,...,2.0,2.0,1.0,1.0,1.0,2.0,1.0,2.7,185.0,25.4
4,Mega Absol,Disaster Pokémon,2 Attack,35 (lower than normal),198,Medium Slow,Field,"50% male, 50% female","25 (6,169–6,425 steps)",65,...,2.0,1.0,0.5,1.0,0.5,1.0,2.0,1.2,49.0,34.0
7,Mega Aerodactyl,Fossil Pokémon,2 Speed,50 (normal),215,Slow,Flying,"87.5% male, 12.5% female","35 (8,739–8,995 steps)",80,...,0.5,2.0,1.0,1.0,1.0,2.0,1.0,2.1,79.0,17.9
9,Mega Aggron,Iron Armor Pokémon,3 Defense,35 (lower than normal),284,Slow,Monster,"50% male, 50% female","35 (8,739–8,995 steps)",70,...,0.5,0.5,1.0,0.5,1.0,0.5,0.5,2.2,395.0,81.6
12,Mega Alakazam,Psi Pokémon,3 Sp. Atk,50 (normal),270,Medium Slow,Human-Like,"75% male, 25% female","20 (4,884–5,140 steps)",55,...,2.0,1.0,2.0,1.0,2.0,1.0,1.0,1.2,48.0,33.3


In [5]:
## Create the NatDex by excluding these duplicates:
NatDex = pokedata.loc[np.logical_not(NDN_duplicates_bool)] # This excludes all duplicates, like Mega's, regionals, and variants

## Now set the National Dex Number as the NatDex DataFrame's index
try:
    NatDex.set_index('National Dex Number', inplace=True, verify_integrity=True)
except:
    print(f'Failed to set National Dex Number as DataFrame Index. Are you sure there are no duplicate National Dex Numbers?')
    pass


C:\Users\jflan\AppData\Local\Temp\ipykernel_40564\2042482982.py:6: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  NatDex.set_index('National Dex Number', inplace=True, verify_integrity=True)


In [6]:
## Order the NatDex to be descending based on index
NatDex.sort_values(by='National Dex Number', ascending=True, inplace=True)

In [7]:
NatDex

,Name,Species,EV Yield,Base Friendship,Base Exp,Growth Rate,Egg Groups,Gender,Egg Cycles,HP Base,...,Against Bug,Against Rock,Against Ghost,Against Dragon,Against Dark,Against Steel,Against Fairy,Height,Weight,BMI
National Dex Number,,,,,,,,,,,,,,,,,,,,,
1,Bulbasaur,Seed Pokémon,1 Sp. Atk,50 (normal),64,Medium Slow,"Grass, Monster","87.5% male, 12.5% female","20 (4,884–5,140 steps)",45,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.7,6.9,14.1
2,Ivysaur,Seed Pokémon,"1 Sp. Atk, 1 Sp. Def",50 (normal),142,Medium Slow,"Grass, Monster","87.5% male, 12.5% female","20 (4,884–5,140 steps)",60,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,1.0,13.0,13.0
3,Venusaur,Seed Pokémon,"2 Sp. Atk, 1 Sp. Def",50 (normal),236,Medium Slow,"Grass, Monster","87.5% male, 12.5% female","20 (4,884–5,140 steps)",80,...,1.0,1.0,1.0,1.0,1.0,1.0,0.5,2.0,100.0,25.0
4,Charmander,Lizard Pokémon,1 Speed,50 (normal),62,Medium Slow,"Dragon, Monster","87.5% male, 12.5% female","20 (4,884–5,140 steps)",39,...,0.5,2.0,1.0,1.0,1.0,0.5,0.5,0.6,8.5,23.6
5,Charmeleon,Flame Pokémon,"1 Sp. Atk, 1 Speed",50 (normal),142,Medium Slow,"Dragon, Monster","87.5% male, 12.5% female","20 (4,884–5,140 steps)",58,...,0.5,2.0,1.0,1.0,1.0,0.5,0.5,1.1,19.0,15.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
894,Regieleki,Electron Pokémon,3 Speed,35 (lower than normal),290,Slow,Undiscovered,Genderless,"120 (30,584–30,840 steps)",80,...,1.0,1.0,1.0,1.0,1.0,0.5,1.0,1.2,145.0,100.7
895,Regidrago,Dragon Orb Pokémon,3 HP,35 (lower than normal),290,Slow,Undiscovered,Genderless,"120 (30,584–30,840 steps)",200,...,1.0,1.0,1.0,2.0,1.0,1.0,2.0,2.1,200.0,45.4
896,Glastrier,Wild Horse Pokémon,3 Attack,35 (lower than normal),290,Slow,Undiscovered,Genderless,"120 (30,584–30,840 steps)",100,...,1.0,2.0,1.0,1.0,1.0,2.0,1.0,2.2,800.0,165.3


## Research Questions
- What is the most common pokemon type? What is the least common?
- What is the most common dual typing (irrespective of the order of Type 1 and Type 2)?

In [9]:
## Group pokemon by primary and secondary type
groupby_type1 = pokedata.groupby(by='Type1')
groupby_type2 = pokedata.groupby(by='Type2')

## Count up the amount of pokemon represented by type
counted_types_1 = groupby_type1.Name.count()
counted_types_2 = groupby_type2.Name.count()

counted_type_df = pd.DataFrame({'Type1' : counted_types_1, 'Type2' : counted_types_2})

## Calculate total counts and percentage of the total for each type
summed_type_data = counted_type_df.sum(axis=1)
percentages = pd.Series(summed_type_data.values/summed_type_data.sum()*100)  # Calculate percentage of total
percentages.index = summed_type_data.index                                   # Set index to equal that of counts


## Put results in new dataframe for clarity
Typedata = pd.DataFrame({'Count' : summed_type_data, 'Percentage' : percentages.round(2)})

print(f'Most common pokemon type: {Typedata.Count.idxmax()} ({Typedata.Percentage.loc[Typedata.Count.idxmax()]} %)')
print(f'Most common pokemon type: {Typedata.Count.idxmin()} ({Typedata.Percentage.loc[Typedata.Count.idxmin()]} %)')

Most common pokemon type: Water (9.81 %)
Most common pokemon type: Ice (3.49 %)


Now, let's see what the most and least common dual typings are.

In [73]:
dual_type_pkmn = pokedata.loc[(pokedata['Type1'].notnull()) & (pokedata['Type2'].notnull())] # select the dual type pkmn

In [74]:
## On its own, the table will treat a T1/T2 combination as distinct from T2/T1; eg. Dragon/Flying is treated
## distinct from Flying/Dragon.

## To get around this, we sort the columns such that Type 1 is always alphabetically 'greater' than Type 2.
## Then, each instance of Flying / Dragon will get re-sorted into Dragon / Flying, and groupby will collect them
## in the way we want.
swap = dual_type_pkmn['Type1'] > dual_type_pkmn['Type2'] # Condition to handle the sorting

## This line addresses all rows in columns Type 1 and Type 2 where the swap condition applies, and replaces those entries with
## the rows from Type 2 and Type 1 where it does *not* apply.
dual_type_pkmn.loc[swap, 'Type1'], dual_type_pkmn.loc[swap, 'Type2'] = dual_type_pkmn['Type2'][swap], dual_type_pkmn['Type1'][swap]

In [114]:
groupby_dualtype = dual_type_pkmn.groupby(['Type1', 'Type2']).size()
total_unique_dualtypes = groupby_dualtype.count()
total_dualtype_pkmn = groupby_dualtype.sum()

print(f'There are {total_dualtype_pkmn} dual-type pokemon, with {total_unique_dualtypes} unique type combinations.')

There are 513 dual-type pokemon, with 132 unique type combinations.


In [130]:
## Most common
dualtype_idxmax = groupby_dualtype.idxmax()
dualtype_max = groupby_dualtype[dualtype_idxmax]

## Least common (trickier as there are many unique combinations; eg. many minima
I = np.nonzero((groupby_dualtype.values - 1)==0) #Returns indices of zeroes, which are the unique combinations
dualtype_min = groupby_dualtype.iloc[I]

In [131]:
print(f'Most common dual typing: {dualtype_idxmax[0]}/{dualtype_idxmax[1]} ({dualtype_max} | {(dualtype_max/total_unique_dualtypes*100).__round__(1)}%)')
print(f'Least common dual typings: ({dualtype_min.values[0]} | {
(dualtype_min.values[0]/total_unique_dualtypes*100).__round__(1)}%)')
for typing in dualtype_min.index:
    print(f'    {typing[0]}/{typing[1]}')

Most common dual typing: Flying/Normal (27 | 20.5%)
Least common dual typings: (1 | 0.8%)
    Bug/Ghost
    Dragon/Fairy
    Dragon/Normal
    Electric/Ghost
    Electric/Ground
    Electric/Ice
    Electric/Poison
    Electric/Psychic
    Fairy/Ghost
    Fairy/Ice
    Fairy/Poison
    Fighting/Ghost
    Fighting/Ice
    Fighting/Rock
    Fire/Steel
    Fire/Water
    Ghost/Ice
    Grass/Ground
    Ground/Normal
    Normal/Water
    Poison/Rock
    Steel/Water
